## Continuing with Andrej Karpathy's exercises to experiment with splitting up the training set

<span style = "color:green;"> E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model? </span>

<span style = "color:orange;"> E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see? </span>

E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

<span style = "color:green;"> E06: meta-exercise! Think of a fun/interesting exercise and complete it. </span>

- added an autoconverging training feature + classes for different Models



**We will split the training set into 80% training, 10% development, and 10% test.**

Firstly,  we import the data in as before.

In [5]:
words = open('names.txt', 'r').read().splitlines()

Similarly we import over some code that will be used to index each character and import pytorch.

In [6]:
import torch
import torch.nn.functional as F

# chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
# stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
# stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
# itos = {i: s for s, i in stoi.items()} # creates a reverse mapping from integer indices back to characters.

I think it makes sense to define Bigram and Trigram classes such that I can call the data handling, gradient descent, and sampling as internal functions.

In [33]:
class BigramLanguageModel:
    def __init__(self, words):
        # creates a random number generator with a fixed seed for reproducibility (The same as used by Karpathy as a consistency check)
        g = torch.Generator().manual_seed(2147483647) 
        self.xs = []
        self.ys = []

        chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
        self.stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
        self.stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
        self.itos = {i: s for s, i in self.stoi.items()} # creates a reverse mapping from integer indices back to characters.

        for w in words:
            chs = ['.'] + list(w) + ['.'] # add start and end tokens
            for ch1, ch2 in zip(chs, chs[1:]):
                self.xs.append(self.stoi[ch1]) # input character index
                self.ys.append(self.stoi[ch2]) # target character index

        self.xs = torch.tensor(self.xs) # convert to tensor
        self.ys = torch.tensor(self.ys)
  
        # Weights for the bigram model, initialized randomly.
        self.W = torch.randn((27, 27), generator=g, requires_grad=True) # 27x27 weight matrix for bigram probabilities (26 letters + '.')


    def forward(self):
        xenc = F.one_hot(self.xs, num_classes=27).float() # one-hot encode the input character indices
        logits = xenc @ self.W # compute logits for the next character
        counts = logits.exp() # convert logits to counts
        self.probs = counts / counts.sum(1, keepdim=True) # normalize counts to get
        return self.probs
    
    def loss(self, regularization_strength=0.01):
        # compute the negative log likelihood loss
        return -self.probs[torch.arange(len(self.ys)), self.ys].log().mean() + regularization_strength * (self.W**2).mean() 
    
    def backward(self):
        self.loss().backward() # compute gradients of the loss with respect to the weights

    def gradient_descent(self, iterations, learning_rate=0.1, verbose=False):
        for i in range(iterations):
            self.forward() # compute probabilities
            self.loss()
            self.W.grad = None
            self.backward() # compute gradients
            self.W.data -= learning_rate * self.W.grad # gradient descent step
            
            if verbose:
                print(f"step {i}: loss = {self.loss().item():.4f}") # print the loss at each step
            return self.loss().item() # return the final loss after training

    def sample(self, num_samples=5):
        g = torch.Generator().manual_seed(2147483647)
        for _ in range(num_samples):
            out = []
            ix = 0  # start token '.'
            while True:
                xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
                logits = xenc @ self.W
                counts = logits.exp()
                probs = counts / counts.sum(1, keepdim=True)
                ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
                if ix == 0:  # end token '.'
                    break
                out.append(self.itos[ix])
            print(''.join(out))

    def train(self, iterations_per_round=50, initial_lr=1.0, min_lr=0.001, decay=0.5, patience=3):
        lr = initial_lr
        best_loss = float('inf')
        rounds_since_improvement = 0
        r = 0

        while True:
            loss_val = self.gradient_descent(iterations=iterations_per_round, learning_rate=lr)
            print(f"round {r:3d}: loss = {loss_val:.4f}  lr = {lr:.5f}")
            r += 1

            if loss_val < best_loss - 1e-4:
                best_loss = loss_val
                rounds_since_improvement = 0
            else:
                rounds_since_improvement += 1

            if rounds_since_improvement >= patience:
                lr *= decay
                rounds_since_improvement = 0
                print(f"  → lr decayed to {lr:.5f}")
                if lr < min_lr:
                    print(f"Converged at loss = {best_loss:.4f}")
                    break



In [ ]:
class TrigramLanguageModel:
    def __init__(self, words):
        # creates a random number generator with a fixed seed for reproducibility (The same as used by Karpathy as a consistency check)
        g = torch.Generator().manual_seed(2147483647) 
        self.xs = []
        self.ys = []

        chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
        self.stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
        self.stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
        self.itos = {i: s for s, i in self.stoi.items()} # creates a reverse mapping from integer indices back to characters.

        for w in words:
            chs = ['.', '.'] + list(w) + ['.'] # add start and end tokens
            for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
                self.xs.append((self.stoi[ch1], self.stoi[ch2])) # input character indices (bigram)
                self.ys.append(self.stoi[ch3]) # target character index

        self.xs = torch.tensor(self.xs) # convert to tensor
        self.ys = torch.tensor(self.ys)
  
        # Weights for the trigram model, initialized randomly.
        self.W = torch.randn((27, 27, 27), generator=g, requires_grad=True) # weight matrix for trigram probabilities (27 x 27 x 27 possible next characters)

    def forward(self):
        xenc1 = F.one_hot(self.xs[:, 0], num_classes=27).float()  # one-hot encode the first character index
        xenc2 = F.one_hot(self.xs[:, 1], num_classes=27).float()  # one-hot encode the second character index
    
        # Split this step into two parts now that we have two input characters.
        # The first part computes the logits for the next character based on the first character encoding and the weight matrix.
        # The second part contracts the logits with the second character encoding to get the final logits for the next character.

        # For each example, we want W[ix1, ix2, :] as the logits.
        # xenc1 @ W reshaped: (N,27) @ (27, 27*27) -> (N, 27*27), then view as (N, 27, 27)
        # then contract with xenc2: sum over the second character dimension

        logits = (xenc1 @ self.W.view(27, 27 * 27)).view(-1, 27, 27)  # compute logits for the next character
        logits = (logits * xenc2.unsqueeze(2)).sum(1)             # contract with the second character encoding to get final logits
        
        counts = logits.exp() # convert logits to counts
        self.probs = counts / counts.sum(1, keepdim=True) # normalize counts to get probabilities
        return self.probs
    
    def loss(self, regularization_strength=0.01):
        # compute the negative log likelihood loss
        return -self.probs[torch.arange(len(self.ys)), self.ys].log().mean() + regularization_strength * (self.W**2).mean()
    
    def backward(self):
        self.loss().backward() # compute gradients of the loss with respect to the weights

    def gradient_descent(self, iterations, learning_rate=0.1, verbose=False):
        for i in range(iterations):
            self.forward() # compute probabilities
            self.loss()
            self.W.grad = None
            self.backward() # compute gradients
            self.W.data -= learning_rate * self.W.grad # gradient descent step
            
            if verbose:
                print(f"step {i}: loss = {self.loss().item():.4f}") # print the loss at each step
            return self.loss().item() # return the final loss after training
    
    def sample(self, num_samples=5):
        g = torch.Generator().manual_seed(2147483647)
        for _ in range(num_samples):
            out = []
            ix1, ix2 = 0, 0  # start tokens '.'
            while True:
                xenc1 = F.one_hot(torch.tensor([ix1]), num_classes=27).float()
                xenc2 = F.one_hot(torch.tensor([ix2]), num_classes=27).float()
                logits = (xenc1 @ self.W.view(27, 27 * 27)).view(-1, 27, 27)
                logits = (logits * xenc2.unsqueeze(2)).sum(1)
                counts = logits.exp()
                probs = counts / counts.sum(1, keepdim=True)
                ix3 = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
                if ix3 == 0:  # end token '.'
                    break
                out.append(self.itos[ix3])
                
                ix1, ix2 = ix2, ix3  # shift the bigram window
            print(''.join(out))

    def train(self, iterations_per_round=50, initial_lr=1.0, min_lr=0.001, decay=0.5, patience=3):
        lr = initial_lr
        best_loss = float('inf')
        rounds_since_improvement = 0
        r = 0

        while True:
            loss_val = self.gradient_descent(iterations=iterations_per_round, learning_rate=lr)
            print(f"round {r:3d}: loss = {loss_val:.4f}  lr = {lr:.5f}")
            r += 1

            if loss_val < best_loss - 1e-4:
                best_loss = loss_val
                rounds_since_improvement = 0
            else:
                rounds_since_improvement += 1

            if rounds_since_improvement >= patience:
                lr *= decay
                rounds_since_improvement = 0
                print(f"  → lr decayed to {lr:.5f}")
                if lr < min_lr:
                    print(f"Converged at loss = {best_loss:.4f}")
                    break

     

In [28]:
BigramModel = BigramLanguageModel(words)
TrigramModel = TrigramLanguageModel(words)


In [34]:
BigramModel = BigramLanguageModel(words)
BigramModel.train(iterations_per_round=100, initial_lr=100.0, min_lr=0.5, patience=3, decay=0.5)

round   0: loss = 3.7673  lr = 100.00000
round   1: loss = 3.1359  lr = 100.00000
round   2: loss = 2.9209  lr = 100.00000
round   3: loss = 2.8069  lr = 100.00000
round   4: loss = 2.7412  lr = 100.00000
round   5: loss = 2.6946  lr = 100.00000
round   6: loss = 2.6636  lr = 100.00000
round   7: loss = 2.6396  lr = 100.00000
round   8: loss = 2.6316  lr = 100.00000
round   9: loss = 2.6173  lr = 100.00000
round  10: loss = 2.6264  lr = 100.00000
round  11: loss = 2.5917  lr = 100.00000
round  12: loss = 2.5855  lr = 100.00000
round  13: loss = 2.5779  lr = 100.00000
round  14: loss = 2.5922  lr = 100.00000
round  15: loss = 2.5633  lr = 100.00000
round  16: loss = 2.5626  lr = 100.00000
round  17: loss = 2.5570  lr = 100.00000
round  18: loss = 2.5735  lr = 100.00000
round  19: loss = 2.5469  lr = 100.00000
round  20: loss = 2.5488  lr = 100.00000
round  21: loss = 2.5444  lr = 100.00000
round  22: loss = 2.5622  lr = 100.00000
round  23: loss = 2.5366  lr = 100.00000
round  24: loss 

In [30]:
TrigramModel.train(iterations_per_round=100, initial_lr=100.0, min_lr=0.5, patience=3, decay=0.5)

round   0: loss = 3.0295  lr = 100.00000
round   1: loss = 2.9923  lr = 100.00000
round   2: loss = 2.9588  lr = 100.00000
round   3: loss = 2.9284  lr = 100.00000
round   4: loss = 2.9007  lr = 100.00000
round   5: loss = 2.8753  lr = 100.00000
round   6: loss = 2.8520  lr = 100.00000
round   7: loss = 2.8306  lr = 100.00000
round   8: loss = 2.8108  lr = 100.00000
round   9: loss = 2.7925  lr = 100.00000
round  10: loss = 2.7754  lr = 100.00000
round  11: loss = 2.7596  lr = 100.00000
round  12: loss = 2.7447  lr = 100.00000
round  13: loss = 2.7308  lr = 100.00000
round  14: loss = 2.7178  lr = 100.00000
round  15: loss = 2.7054  lr = 100.00000
round  16: loss = 2.6938  lr = 100.00000
round  17: loss = 2.6827  lr = 100.00000
round  18: loss = 2.6723  lr = 100.00000
round  19: loss = 2.6623  lr = 100.00000
round  20: loss = 2.6528  lr = 100.00000
round  21: loss = 2.6437  lr = 100.00000
round  22: loss = 2.6350  lr = 100.00000
round  23: loss = 2.6267  lr = 100.00000
round  24: loss 

In [37]:
BigramModel.sample(num_samples=10)
TrigramModel.sample(num_samples=10)

cexze
momasurailezityha
konimittain
llayn
ka
da
staiyaubrtthrigotai
moliellavo
ke
teda
ce
bra
jalius
rochityharlonimittain
luwan
ka
da
samiyah
javer
gotai
